In [1]:
import numpy as np
import matplotlib
from matplotlib import pyplot as plot
from sklearn.linear_model import LinearRegression
import sklearn.metrics
from sklearn.metrics import explained_variance_score, mean_squared_error, r2_score # https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols.html
import statsmodels.api as sm
import pandas
import numpy as np
import matplotlib.pyplot as plt
import pandas

# For 3d plots. This import is necessary to have 3D plotting below
from mpl_toolkits.mplot3d import Axes3D

# For statistics. Requires statsmodels 5.0 or more
from statsmodels.formula.api import ols
# Analysis of Variance (ANOVA) on linear models
from statsmodels.stats.anova import anova_lm

# Adapted from:
# https://www.datasciencecentral.com/profiles/blogs/linear-regression-in-python-use-of-numpy-scipy-and-statsmodels,
# https://realpython.com/linear-regression-in-python/#multiple-linear-regression


In [2]:
DataToFit = np.transpose(np.asarray(np.genfromtxt('deltat5e8forHenry.csv',delimiter=',',skip_header=1)))
# 6 = logM
logMIndex = 6
# 7 = logL
logLIndex = 7
# 9 = logR
logRIndex = 9
# 14 = Fundamental period from Trabucchi model
P0Index = 13
# 15 = logP0 from Trabucchi model
logP0Index = 12
# 15 = Overtone period from Trabucchi model
P1Index=15
# 15 = logP1 from COLIBRI model
logP1Index=16
# 17 = Pulsation Mode
ModeIndex=17
# 25 = log(C/O)
logCORatioIndex=25

In [3]:
DataToFit[10]

array([ 0.449,  0.447,  0.477,  0.477,  0.461,  0.463,  0.46 ,  0.447,
        0.442,  0.465,  0.469,  0.253,  0.251,  0.383,  0.454,  0.485,
        0.272,  0.475,  0.391,  0.458,  0.466,  0.377,  0.369,  0.331,
        0.437,  0.465,  0.357,  0.629,  0.475,  0.275,  0.437,  0.243,
        0.62 ,  0.46 ,  0.259,  0.425,  0.341,  0.491,  0.387,  0.417,
        0.629,  0.294,  0.447,  0.196,  0.2  ,  0.219,  0.423,  0.326,
        0.34 ,  0.211,  0.27 ,  0.039,  0.319,  0.531,  0.254,  0.473,
        0.267,  0.238,  0.04 ,  0.518,  0.366,  0.508,  0.401,  0.522,
        0.451,  0.432,  0.257,  0.49 ,  0.483,  0.44 ,  0.426,  0.059,
        0.265,  0.298,  0.324,  0.138,  0.387,  0.453,  0.123,  0.276,
        0.433,  0.187,  0.165,  0.416,  0.422,  0.184,  0.16 ,  0.324,
        0.135,  0.512,  0.489,  0.48 ,  0.216,  0.162,  0.407,  0.257,
        0.49 ,  0.424,  0.697,  0.32 ,  0.139,  0.268,  0.26 ,  0.41 ,
        0.281,  0.342,  0.193, -0.099,  0.039,  0.121,  0.342,  0.195,
      

In [4]:
def PulsationModeSort (input_array):
    FundamentalPulsators = []
    OvertonePulsators = []
    i = 0
    while i < len(input_array[ModeIndex]):
        if input_array[ModeIndex][i] == 0:
            FundamentalPulsators.append(i)
        if input_array[ModeIndex][i] == 1:
            OvertonePulsators.append(i)
        i += 1
    FundamentalModeStars = np.transpose(input_array.transpose()[FundamentalPulsators])
    FirstOvertoneModeStars = np.transpose(input_array.transpose()[OvertonePulsators])
    return FundamentalModeStars, FirstOvertoneModeStars

def CompositionSort (input_array):
    ORichStarsIndices = []
    CRichStarsIndices = []
    i = 0
    while i < len(input_array[logCORatioIndex]):
        if input_array[logCORatioIndex][i] < 0:
            ORichStarsIndices.append(i)
        if input_array[logCORatioIndex][i] > 0:
            CRichStarsIndices.append(i)
        i += 1
    ORichStars = np.transpose(input_array.transpose()[ORichStarsIndices])
    CRichStars = np.transpose(input_array.transpose()[CRichStarsIndices])
    return ORichStars, CRichStars

In [5]:
P0Stars, P1Stars = PulsationModeSort(DataToFit)

#ORichP0Stars, CRichP0Stars = CompositionSort(P0Stars)
#ORichP1Stars, CRichP1Stars = CompositionSort(P1Stars)

In [6]:
# First, let's look at all of the Fundamental mode data at once.
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = P0Stars[logMIndex]
logRData = P0Stars[logRIndex]
logPData = P0Stars[logP0Index]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.998
Model:                            OLS   Adj. R-squared:                  0.998
Method:                 Least Squares   F-statistic:                 9.441e+04
Date:                Thu, 06 Feb 2020   Prob (F-statistic):               0.00
Time:                        09:47:05   Log-Likelihood:                 1092.6
No. Observations:                 357   AIC:                            -2179.
Df Residuals:                     354   BIC:                            -2168.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.3575      0.012   -194.824      0.0

C Stars: logP0 = -2.358(12) - 0.887(6)logM + 2.077(5)logR

In [13]:
# Now, let's look at all of the first overtone mode data at once.
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logMData = P1Stars[logMIndex]
logRData = P1Stars[logRIndex]
logPData = P1Stars[logP1Index]

data = pandas.DataFrame({'x0': logMData, 'x1': logRData, 'y': logPData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.896
Model:                            OLS   Adj. R-squared:                  0.895
Method:                 Least Squares   F-statistic:                     1786.
Date:                Thu, 06 Feb 2020   Prob (F-statistic):          6.78e-205
Time:                        00:10:05   Log-Likelihood:                 611.67
No. Observations:                 419   AIC:                            -1217.
Df Residuals:                     416   BIC:                            -1205.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.2114      0.066    -18.432      0.0

C stars: logP1 = -1.211(66) - 0.302(32)logM + 1.407(32)logR

In [14]:
# Now, let's look at all of the first overtone mode data at once.
# Convert the data into a Pandas DataFrame to use the formulas framework
# in statsmodels

# First we need to flatten the data: it's 2D layout is not relevent.
logRData = DataToFit[logRIndex]
logLData = DataToFit[logLIndex]
logMData = DataToFit[logMIndex]

data = pandas.DataFrame({'x0': logLData, 'x1': logMData, 'y': logRData})

# Fit the model
model = ols("y ~ x0 + x1", data).fit()

# Print the summary
print(model.summary())

print("\nRetrieving manually the parameter estimates:")
print(model._results.params)

# Peform analysis of variance on fitted linear model
anova_results = anova_lm(model)

print('\nANOVA results')
print(anova_results)

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.935
Model:                            OLS   Adj. R-squared:                  0.935
Method:                 Least Squares   F-statistic:                     5558.
Date:                Thu, 06 Feb 2020   Prob (F-statistic):               0.00
Time:                        00:10:55   Log-Likelihood:                 1335.6
No. Observations:                 776   AIC:                            -2665.
Df Residuals:                     773   BIC:                            -2651.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4095      0.027    -14.957      0.0

C Stars: logR = -0.410(27) + 0.744(8)logL - 0.246(12)logM